# NUCLEAR — Standalone EVO + cross-task improvement add-in

This is a **true standalone notebook**. Open it in a fresh Colab runtime; do **not** append it to the main Nuclear notebook.

It:
- mounts Drive and rebuilds only the lightweight GEAR context;
- reuses cached nuclear explicit features;
- reconstructs V1/MS OOF predictions directly from saved fold prediction files;
- exports Nuclear OOF for the Small standalone notebook;
- waits for the Small export;
- tests EVO, EVO residual beyond ACCESS, cross-task Small OOF probabilities, and combinations using Dev-only MLP CV.

No PDB normalization or V1/MS retraining is required.

In [ ]:
# ============================================================
# STANDALONE BOOTSTRAP — fresh Colab runtime
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, sys, json, ast, time, subprocess
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import average_precision_score

BASE = Path('/content/drive/MyDrive/Protein-binding-Phi')

# Lightweight requirements only.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "numpy", "pandas", "scipy", "scikit-learn", "tqdm"],
    check=True
)

ULTIMATE_CANDIDATES = [
    BASE/'GEAR_ULTIMATE_OTHER_BINDINGS',
    BASE/'GEAR_ULTIMATE',
]
ULTIMATE_DIR = next(
    (p for p in ULTIMATE_CANDIDATES if (p/'gear_ultimate').exists()),
    None
)
if ULTIMATE_DIR is None:
    raise FileNotFoundError(
        'GEAR backend not found under '
        'GEAR_ULTIMATE_OTHER_BINDINGS or GEAR_ULTIMATE'
    )

req = ULTIMATE_DIR/'requirements.txt'
if req.exists():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(req)],
        check=True
    )

sys.path.insert(0, str(ULTIMATE_DIR))

from gear_ultimate import gear_evolution as evo
from gear_ultimate import gear_suite as suite
from gear_ultimate import gear_reporting as reporting

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    torch.set_float32_matmul_precision('high')

print('BASE:', BASE)
print('GEAR backend:', ULTIMATE_DIR)
print('DEVICE:', DEVICE)


Mounted at /content/drive
BASE: /content/drive/MyDrive/Protein-binding-Phi
GEAR backend: /content/drive/MyDrive/Protein-binding-Phi/GEAR_ULTIMATE_OTHER_BINDINGS
DEVICE: cpu


In [ ]:
# Standalone settings
# These notebooks are designed for TWO FRESH COLAB RUNTIMES.
# Launch Nuclear and Small at roughly the same time.
# Each exports its Dev OOF early, then waits for the counterpart.
print("Standalone add-in runtime started.")


Standalone add-in runtime started.


In [ ]:
# ============================================================
# DATA / OFFICIAL FOLDS / STORE HELPERS
# ============================================================
TRAIN_CSV = BASE/'data/development_set/full_grouped_train_binding_sites_df.csv'
TEST300_CSV = BASE/'data/development_set/full_grouped_test_binding_sites_df.csv'
TEST46_CSV = BASE/'data/independent_set/grouped_test_46_new_binding_sites.csv'

def parse_list(v):
    if isinstance(v, str):
        return list(ast.literal_eval(v))
    return list(v)

def load_df(path):
    df = pd.read_csv(path)
    df['prot_id'] = df['prot_id'].astype(str)
    df['sequence'] = df['sequence'].astype(str)
    for col in [
        'metal_binding', 'nuclear_binding',
        'small_binding', 'any_ligand_binding_sites'
    ]:
        if col in df.columns:
            df[col] = df[col].apply(parse_list)
    return df

raw_train = load_df(TRAIN_CSV)
test300_df = load_df(TEST300_CSV)
test46_df = load_df(TEST46_CSV)

BASELINE_WORK = BASE/'baseline_reproduction_master_v6'
BN24_CANDIDATES = [
    BASELINE_WORK/'repos'/'bindnode24',
    BASE/'repos'/'bindnode24',
]
BN24 = next((p for p in BN24_CANDIDATES if (p/'.git').exists()), None)

if BN24 is None:
    BN24 = Path('/content/bindnode24_standalone_addin')
    if not BN24.exists():
        subprocess.run(
            ['git','clone','-q','https://github.com/erckert/bindNode24.git',str(BN24)],
            check=True
        )

fold_root = BN24/'dataset'/'cv_splits'/'predicted_structures'
OFFICIAL_FOLDS = {}
official_order = []
seen = set()

for k in range(1,6):
    ids = [
        x.strip()
        for x in (fold_root/f'ids_split{k}.txt').read_text().splitlines()
        if x.strip()
    ]
    if seen.intersection(ids):
        raise RuntimeError(f'Fold overlap at fold {k}')
    OFFICIAL_FOLDS[k] = ids
    official_order.extend(ids)
    seen.update(ids)

train_lookup = raw_train.set_index('prot_id', drop=False)
missing = set(official_order) - set(raw_train.prot_id)
if missing:
    raise RuntimeError(f'Missing official Dev IDs: {sorted(missing)[:20]}')

train_df = train_lookup.loc[official_order].reset_index(drop=True)

assert len(train_df) == 1010
assert len(test300_df) == 300
assert len(test46_df) == 46

SPLIT_DF = {
    'DevSet1010': train_df,
    'TestSet300': test300_df,
    'TestSetNew46': test46_df,
}
STORE_KEY = {
    'DevSet1010': 'train',
    'TestSet300': 'test300',
    'TestSetNew46': 'test46',
}

def get_record(store, pid):
    pid = str(pid)
    if pid in store:
        return store[pid]
    raise KeyError(pid)

def get_block(store, pid, block):
    rec = get_record(store, pid)

    if isinstance(rec, dict):
        if block in rec:
            return np.asarray(rec[block])
        if isinstance(rec.get('features'), dict) and block in rec['features']:
            return np.asarray(rec['features'][block])

    if hasattr(rec, block):
        return np.asarray(getattr(rec, block))

    for attr in ('features','feature_dict','blocks'):
        d = getattr(rec, attr, None)
        if isinstance(d, dict) and block in d:
            return np.asarray(d[block])

    raise KeyError(f'{pid}/{block}')

def set_block(store, pid, block, arr):
    rec = get_record(store, pid)
    arr = np.asarray(arr, dtype=np.float32)

    if isinstance(rec, dict):
        if isinstance(rec.get('features'), dict):
            rec['features'][block] = arr
        else:
            rec[block] = arr
        return

    for attr in ('features','feature_dict','blocks'):
        d = getattr(rec, attr, None)
        if isinstance(d, dict):
            d[block] = arr
            return

    setattr(rec, block, arr)

def reconstruct_oof_from_fold_predictions(pred_dir, label_col):
    """
    Saved foldX_val.npy was produced on:
      va_df = train_df[train_df.prot_id.isin(val_set)]
    preserving train_df order.
    """
    offsets = {}
    cursor = 0
    for r in train_df.itertuples():
        L = len(r.sequence)
        offsets[str(r.prot_id)] = (cursor, cursor+L)
        cursor += L

    oof = np.full(cursor, np.nan, dtype=np.float32)

    for fold, val_ids in OFFICIAL_FOLDS.items():
        val_set = set(val_ids)
        va_df = train_df[train_df.prot_id.isin(val_set)].copy()

        fp = Path(pred_dir)/f'fold{fold}_val.npy'
        if not fp.exists():
            raise FileNotFoundError(fp)

        pred = np.load(fp).astype(np.float32)
        expected = int(va_df.sequence.str.len().sum())

        if pred.shape != (expected,):
            raise RuntimeError(
                f'{fp}: shape {pred.shape}, expected {(expected,)}'
            )

        pos = 0
        for r in va_df.itertuples():
            L = len(r.sequence)
            a,b = offsets[str(r.prot_id)]
            oof[a:b] = pred[pos:pos+L]
            pos += L

    if not np.all(np.isfinite(oof)):
        raise RuntimeError('Incomplete reconstructed OOF')

    y = np.concatenate(
        train_df[label_col].apply(
            lambda x: np.asarray(x, dtype=np.int8)
        ).tolist()
    )

    return oof, y

def official_fold_indices():
    offsets = {}
    cursor = 0
    for r in train_df.itertuples():
        L = len(r.sequence)
        offsets[str(r.prot_id)] = (cursor,cursor+L)
        cursor += L

    out = {}
    for fold, ids in OFFICIAL_FOLDS.items():
        z = []
        for pid in ids:
            a,b = offsets[str(pid)]
            z.extend(range(a,b))
        out[fold] = np.asarray(z, dtype=np.int64)
    return out

FOLD_IDX = official_fold_indices()

def weight_search(y, p1, p2):
    rows = []
    for w in np.linspace(0,1,21):
        p = w*p1 + (1-w)*p2
        aps = [
            average_precision_score(y[idx], p[idx])
            for _,idx in sorted(FOLD_IDX.items())
        ]
        rows.append({
            'w_V1': float(w),
            'w_MS': float(1-w),
            'CV_AUPRC_mean': float(np.mean(aps)),
            'CV_AUPRC_std': float(np.std(aps, ddof=1)),
            'OOF_AUPRC': float(average_precision_score(y,p)),
            **{f'fold{i+1}_AUPRC': float(v) for i,v in enumerate(aps)}
        })
    return pd.DataFrame(rows).sort_values(
        ['CV_AUPRC_mean','w_V1'],
        ascending=[False,False]
    ).reset_index(drop=True)

print('Dataset/fold bootstrap: PASS')


Dataset/fold bootstrap: PASS


In [ ]:
# ============================================================
# NUCLEAR STANDALONE CONTEXT + CACHED EXPLICIT BLOCKS + EVO
# ============================================================
ADDIN_ROOT = BASE/'results'/'ADDIN_STANDALONE_NUCLEAR_EVO_XTASK'
ADDIN_ROOT.mkdir(parents=True, exist_ok=True)

XTASK_ROOT = BASE/'results'/'CROSSTASK_EXPORTS'
XTASK_ROOT.mkdir(parents=True, exist_ok=True)

ctx = suite.prepare_context(
    base=BASE,
    run_version='standalone_nuclear_addin_context',
    label_column='nuclear_binding',
    models=('MLP',),
    ensure_base=True,
    add_mech=True,
    add_multi=True,
    add_seqctx=True,
    esm_batch_size=8,
    device=DEVICE,
)

for split, df in SPLIT_DF.items():
    store_ids = set(map(str, ctx.stores[STORE_KEY[split]].keys()))
    csv_ids = set(df.prot_id.astype(str))
    if store_ids != csv_ids:
        raise RuntimeError(f'{split}: store/CSV mismatch')

# Attach cached nuclear explicit blocks directly. No PDB rebuild.
FEATURE_CACHE = BASE/'cache'/'nuclear_ultimate_v1'/'nuclear_explicit_v1'
NEW_BLOCKS = {
    'NA_SEQELECTRO32':32,
    'NA_PATCH3D40':40,
    'NA_HBOND16':16,
    'NA_TOPO12':12,
    'NA_PROTCTX8':8,
    'NA_AROMCTRL8':8,
}

for split, df in SPLIT_DF.items():
    store = ctx.stores[STORE_KEY[split]]
    outdir = FEATURE_CACHE/split

    for r in df.itertuples():
        fp = outdir/f'{r.prot_id}.npz'
        if not fp.exists():
            raise FileNotFoundError(
                f'Missing cached nuclear feature file: {fp}'
            )
        z = np.load(fp)
        for k,d in NEW_BLOCKS.items():
            a = z[k].astype(np.float32, copy=False)
            if a.shape != (len(r.sequence), d):
                raise RuntimeError(f'{split}/{r.prot_id}/{k}: {a.shape}')
            set_block(store, r.prot_id, k, a)

print('Cached nuclear explicit blocks attached.')

# Attach cached EVO/MSA if available.
EVO_AVAILABLE = False
try:
    coverage = evo.attach_evo(
        ctx.stores,
        BASE,
        splits=('train','test300','test46'),
        require_msa=True,
    )
    display(coverage)
    EVO_AVAILABLE = True
    print('EVO attached.')
except Exception as e:
    print('EVO unavailable -> experiment will continue without EVO.')
    print(repr(e))

display(reporting.dimension_table(ctx.stores['train']))


[DevSet1010] Reused ESM cache: DevSet1010_ESM2_t33_mean_last2_1280_fp16.pkl
[DevSet1010] Reused structure cache: DevSet1010_spatial_local_v2.pkl
[TestSet300] Reused ESM cache: TestSet300_ESM2_t33_mean_last2_1280_fp16.pkl
[TestSet300] Reused structure cache: TestSet300_spatial_local_v2.pkl
[TestSetNew46] Reused ESM cache: TestSetNew46_ESM2_t33_mean_last2_1280_fp16.pkl
[TestSetNew46] Reused structure cache: TestSetNew46_spatial_local_v2.pkl
[DevSet1010] Reused ATOM cache: DevSet1010_atomic_coordination_v1.pkl
[DevSet1010] Reused CHEM cache: DevSet1010_chemistry_v1_24d.pkl
[TestSet300] Reused ATOM cache: TestSet300_atomic_coordination_v1.pkl
[TestSet300] Reused CHEM cache: TestSet300_chemistry_v1_24d.pkl
[TestSetNew46] Reused ATOM cache: TestSetNew46_atomic_coordination_v1.pkl
[TestSetNew46] Reused CHEM cache: TestSetNew46_chemistry_v1_24d.pkl
[DevSet1010] Reused MECH1 cache: DevSet1010_mechanistic_v1.pkl
[DevSet1010] Reused MULTI2 cache: DevSet1010_multisignal_v2.pkl
[TestSet300] Reused 

EVO cache:   0%|          | 0/1353 [00:00<?, ?it/s]

,split,protein,msa_depth,has_real_msa
0,train,A0A0F7RDM3,977,True
1,train,A0A0F7RHX8,1024,True
2,train,A0A0H2W6Y8,1023,True
3,train,A0A2B6C3P9,1024,True
4,train,A0A384LKY8,542,True
...,...,...,...,...
1351,test46,Q9KDJ7,1023,True
1352,test46,Q9LFM3,1018,True
1353,test46,Q9SJ89,1022,True
1354,test46,U2EQ00,1021,True


EVO attached.


,block,dim
0,AA,20
6,ACCESS,10
4,ATOM,15
5,CHEM,24
9,CONF,4
11,ELEC3D,8
8,ENV,8
1,ESM,1280
22,EVO,48
3,LOCAL,8


In [ ]:
# ============================================================
# RECONSTRUCT NUCLEAR V1/MS OOF FROM SAVED FOLD PREDICTIONS
# EXPORT FOR SMALL STANDALONE NOTEBOOK
# ============================================================
V1_PRED_DIR = (
    BASE/'results'/'NUCLEAR_ULTIMATE_V1_DATA_DRIVEN_PATCH'
    /'patch_states'/'NA_PatchHG_Attention'
)

MS_PRED_DIR = (
    BASE/'results'/'NUCLEAR_V1_ATTENTION_V2_MULTISCALE_PATCH'
    /'states'/'NA_PatchHG_Attention_MS'
)

V1_OOF, Y_DEV = reconstruct_oof_from_fold_predictions(
    V1_PRED_DIR, 'nuclear_binding'
)
MS_OOF, Y2 = reconstruct_oof_from_fold_predictions(
    MS_PRED_DIR, 'nuclear_binding'
)
assert np.array_equal(Y_DEV, Y2)

ws = weight_search(Y_DEV, V1_OOF, MS_OOF)
display(ws.head(10))
ws.to_csv(ADDIN_ROOT/'DEV_WEIGHT_SEARCH.csv', index=False)

best = ws.iloc[0]
W_V1 = float(best.w_V1)
W_MS = float(best.w_MS)
FINAL_OOF = W_V1*V1_OOF + W_MS*MS_OOF

print('Locked weights:', W_V1, W_MS)
print('Dev mean AUPRC:', float(best.CV_AUPRC_mean))

rows = []
pos = 0
for r in train_df.itertuples():
    L = len(r.sequence)
    y = np.asarray(r.nuclear_binding, dtype=np.int8)
    for i in range(L):
        rows.append({
            'dataset':'DevSet1010',
            'prot_id':str(r.prot_id),
            'residue_index':i,
            'y_nuclear':int(y[i]),
            'prob_FINAL':float(FINAL_OOF[pos+i]),
            'prob_V1':float(V1_OOF[pos+i]),
            'prob_MS':float(MS_OOF[pos+i]),
        })
    pos += L

export = pd.DataFrame(rows)
export_fp = XTASK_ROOT/'NUCLEAR_DEV_OOF.csv.gz'
tmp_fp = XTASK_ROOT/'NUCLEAR_DEV_OOF.csv.gz.tmp'
export.to_csv(tmp_fp, index=False)
os.replace(tmp_fp, export_fp)

print('Nuclear OOF export:', export_fp)


,w_V1,w_MS,CV_AUPRC_mean,CV_AUPRC_std,OOF_AUPRC,fold1_AUPRC,fold2_AUPRC,fold3_AUPRC,fold4_AUPRC,fold5_AUPRC
0,0.70,0.30,0.508961,0.088127,0.499399,0.621704,0.569264,0.416005,0.506799,0.431034
1,0.75,0.25,0.508836,0.088383,0.498268,0.622199,0.569397,0.415494,0.505746,0.431343
2,0.60,0.40,0.508830,0.087920,0.500777,0.620899,0.568362,0.415829,0.508760,0.430301
3,0.80,0.20,0.508746,0.088352,0.496696,0.622117,0.569541,0.414385,0.504810,0.432879
4,0.65,0.35,0.508732,0.088106,0.500164,0.621396,0.568819,0.415991,0.507109,0.430348
5,0.55,0.45,0.508344,0.088076,0.500859,0.620327,0.567899,0.414923,0.509061,0.429511
6,0.85,0.15,0.508255,0.088613,0.494368,0.622004,0.569522,0.412994,0.503455,0.433299
7,0.50,0.50,0.507818,0.088120,0.500562,0.619743,0.566955,0.414553,0.509612,0.428228
8,0.90,0.10,0.507795,0.088558,0.490964,0.621642,0.569000,0.411015,0.502073,0.435245
9,0.45,0.55,0.507183,0.088312,0.499897,0.619011,0.566335,0.413627,0.509952,0.426988


Locked weights: 0.7000000000000001 0.29999999999999993
Dev mean AUPRC: 0.5089611583669623
Nuclear OOF export: /content/drive/MyDrive/Protein-binding-Phi/results/CROSSTASK_EXPORTS/NUCLEAR_DEV_OOF.csv.gz


FIXED: /content/drive/MyDrive/Protein-binding-Phi/results/CROSSTASK_EXPORTS/SMALL_DEV_OOF.csv.gz
rows: 170181
columns: ['dataset', 'prot_id', 'residue_index', 'y_nuclear', 'prob_FINAL', 'prob_V1', 'prob_MS']
      dataset prot_id  residue_index  y_nuclear  prob_FINAL   prob_V1  \
0  DevSet1010  Q5LL55              0          0    0.003847  0.004833   
1  DevSet1010  Q5LL55              1          0    0.169308  0.235908   
2  DevSet1010  Q5LL55              2          0    0.076046  0.108088   
3  DevSet1010  Q5LL55              3          0    0.028596  0.040565   
4  DevSet1010  Q5LL55              4          0    0.027697  0.038627   

    prob_MS  
0  0.001546  
1  0.013907  
2  0.001280  
3  0.000668  
4  0.002193  


In [ ]:
# ============================================================
# FIX NUCLEAR OOF EXPORT — WRITE REAL GZIP
# ============================================================
from pathlib import Path
import os
import pandas as pd

XTASK_ROOT = BASE / 'results' / 'CROSSTASK_EXPORTS'
export_fp = XTASK_ROOT / 'NUCLEAR_DEV_OOF.csv.gz'
tmp_fp = XTASK_ROOT / 'NUCLEAR_DEV_OOF.tmp.csv.gz'

assert 'export' in globals(), "Need the previous Nuclear OOF export cell state."

export.to_csv(
    tmp_fp,
    index=False,
    compression='gzip'
)

os.replace(
    tmp_fp,
    export_fp
)

_check = pd.read_csv(export_fp)

print("FIXED:", export_fp)
print("rows:", len(_check))
print("columns:", _check.columns.tolist())
print(_check.head())

FIXED: /content/drive/MyDrive/Protein-binding-Phi/results/CROSSTASK_EXPORTS/NUCLEAR_DEV_OOF.csv.gz
rows: 170181
columns: ['dataset', 'prot_id', 'residue_index', 'y_nuclear', 'prob_FINAL', 'prob_V1', 'prob_MS']
      dataset prot_id  residue_index  y_nuclear  prob_FINAL   prob_V1  \
0  DevSet1010  Q5LL55              0          0    0.003847  0.004833   
1  DevSet1010  Q5LL55              1          0    0.169308  0.235908   
2  DevSet1010  Q5LL55              2          0    0.076046  0.108088   
3  DevSet1010  Q5LL55              3          0    0.028596  0.040565   
4  DevSet1010  Q5LL55              4          0    0.027697  0.038627   

    prob_MS  
0  0.001546  
1  0.013907  
2  0.001280  
3  0.000668  
4  0.002193  


In [ ]:
# ============================================================
# WAIT FOR SMALL EXPORT -> ATTACH X-TASK + EVO RESIDUAL -> SCREEN
# ============================================================
COUNTERPART = XTASK_ROOT/'SMALL_DEV_OOF.csv.gz'
WAIT_SECONDS = 900
POLL_SECONDS = 10

t0 = time.time()
while not COUNTERPART.exists() and (time.time()-t0) < WAIT_SECONDS:
    print(
        f'Waiting for Small export... '
        f'{int(time.time()-t0)}s / {WAIT_SECONDS}s',
        end='\r'
    )
    time.sleep(POLL_SECONDS)

XTASK_SMALL_AVAILABLE = COUNTERPART.exists()
print('\nSmall cross-task export available:', XTASK_SMALL_AVAILABLE)

if XTASK_SMALL_AVAILABLE:
    xdf = pd.read_csv(COUNTERPART)
    xdf['prot_id'] = xdf['prot_id'].astype(str)

    store = ctx.stores['train']
    for r in train_df.itertuples():
        pid = str(r.prot_id)
        sub = xdf[xdf.prot_id.eq(pid)].sort_values('residue_index')
        if len(sub) != len(r.sequence):
            raise RuntimeError(f'{pid}: cross-task length mismatch')
        arr = sub[['prob_FINAL','prob_V1','prob_MS']].to_numpy(np.float32)
        set_block(store, pid, 'XTASK_SMALL3', arr)

# EVO residual beyond ACCESS/RSA-like information.
if EVO_AVAILABLE:
    for split, df in SPLIT_DF.items():
        store = ctx.stores[STORE_KEY[split]]

        for r in df.itertuples():
            E = np.asarray(get_block(store, r.prot_id, 'EVO'), dtype=np.float64)
            A = np.asarray(get_block(store, r.prot_id, 'ACCESS'), dtype=np.float64)

            if A.ndim == 1:
                A = A[:,None]

            X = np.concatenate(
                [np.ones((len(A),1)), A, A*A],
                axis=1
            )

            if len(E) > X.shape[1] + 2:
                beta = np.linalg.lstsq(X, E, rcond=None)[0]
                R = E - X @ beta
            else:
                R = E - E.mean(axis=0, keepdims=True)

            sd = R.std(axis=0, keepdims=True)
            R = R / np.where(sd < 1e-6, 1.0, sd)
            set_block(store, r.prot_id, 'EVO_ACCESS_RES', R.astype(np.float32))

BASE_BLOCKS = [
    'ESM',
    'ACCESS',
    'NA_PATCH3D40',
    'NA_TOPO12',
]

GRID = {
    'N_BASE_PATCH_ORGANIZATION': BASE_BLOCKS,
}

if EVO_AVAILABLE:
    GRID['N_BASE_PLUS_EVO'] = BASE_BLOCKS + ['EVO']
    GRID['N_BASE_PLUS_EVO_ACCESS_RES'] = BASE_BLOCKS + ['EVO_ACCESS_RES']
    GRID['N_BASE_PLUS_EVO_BOTH'] = BASE_BLOCKS + ['EVO','EVO_ACCESS_RES']

if XTASK_SMALL_AVAILABLE:
    GRID['N_BASE_PLUS_XTASK_SMALL'] = BASE_BLOCKS + ['XTASK_SMALL3']
    if EVO_AVAILABLE:
        GRID['N_BASE_PLUS_EVORES_XTASK_SMALL'] = (
            BASE_BLOCKS + ['EVO_ACCESS_RES','XTASK_SMALL3']
        )

print('Screening:')
for k,v in GRID.items():
    print(' ',k,'=',v)

cv, folds = suite.run_dev_cv_only(
    ctx,
    GRID,
    models=('MLP',),
    run_version='standalone_nuclear_evo_xtask_screen',
)

base = float(
    cv.loc[
        cv.representation.eq('N_BASE_PATCH_ORGANIZATION'),
        'CV_AUPRC'
    ].iloc[0]
)

cv = cv.copy()
cv['delta_vs_base'] = cv['CV_AUPRC'] - base
cv = cv.sort_values('CV_AUPRC', ascending=False).reset_index(drop=True)

display(cv)
display(folds)

cv.to_csv(ADDIN_ROOT/'REPRESENTATION_SCREEN.csv', index=False)
folds.to_csv(ADDIN_ROOT/'REPRESENTATION_SCREEN_FOLDS.csv', index=False)

best = cv.iloc[0]
decision = {
    'base_dev_auprc': base,
    'best_representation': str(best.representation),
    'best_dev_auprc': float(best.CV_AUPRC),
    'gain_vs_base': float(best.delta_vs_base),
    'promote_if_gain_ge_0.002': bool(best.delta_vs_base >= 0.002),
    'EVO_available': bool(EVO_AVAILABLE),
    'XTASK_small_available': bool(XTASK_SMALL_AVAILABLE),
}
(ADDIN_ROOT/'DECISION.json').write_text(json.dumps(decision, indent=2))

print('\nFINAL DECISION')
print(json.dumps(decision, indent=2))
print('Saved:', ADDIN_ROOT)



Small cross-task export available: True
Screening:
  N_BASE_PATCH_ORGANIZATION = ['ESM', 'ACCESS', 'NA_PATCH3D40', 'NA_TOPO12']
  N_BASE_PLUS_EVO = ['ESM', 'ACCESS', 'NA_PATCH3D40', 'NA_TOPO12', 'EVO']
  N_BASE_PLUS_EVO_ACCESS_RES = ['ESM', 'ACCESS', 'NA_PATCH3D40', 'NA_TOPO12', 'EVO_ACCESS_RES']
  N_BASE_PLUS_EVO_BOTH = ['ESM', 'ACCESS', 'NA_PATCH3D40', 'NA_TOPO12', 'EVO', 'EVO_ACCESS_RES']
  N_BASE_PLUS_XTASK_SMALL = ['ESM', 'ACCESS', 'NA_PATCH3D40', 'NA_TOPO12', 'XTASK_SMALL3']
  N_BASE_PLUS_EVORES_XTASK_SMALL = ['ESM', 'ACCESS', 'NA_PATCH3D40', 'NA_TOPO12', 'EVO_ACCESS_RES', 'XTASK_SMALL3']
REUSE MLP N_BASE_PATCH_ORGANIZATION 1
REUSE MLP N_BASE_PATCH_ORGANIZATION 2
REUSE MLP N_BASE_PATCH_ORGANIZATION 3
REUSE MLP N_BASE_PATCH_ORGANIZATION 4
REUSE MLP N_BASE_PATCH_ORGANIZATION 5
REUSE MLP N_BASE_PLUS_EVO 1
REUSE MLP N_BASE_PLUS_EVO 2
REUSE MLP N_BASE_PLUS_EVO 3
REUSE MLP N_BASE_PLUS_EVO 4
REUSE MLP N_BASE_PLUS_EVO 5
REUSE MLP N_BASE_PLUS_EVO_ACCESS_RES 1
REUSE MLP N_BASE_PLUS_EVO_AC

,model,representation,CV_AUPRC,CV_AUPRC_SD,CV_AUROC,delta_vs_base
0,MLP,N_BASE_PLUS_EVO,0.478964,0.073589,0.952476,0.007125
1,MLP,N_BASE_PLUS_EVO_ACCESS_RES,0.477783,0.075010,0.952847,0.005944
2,MLP,N_BASE_PLUS_EVORES_XTASK_SMALL,0.475209,0.092198,0.949933,0.003369
3,MLP,N_BASE_PLUS_EVO_BOTH,0.472256,0.084920,0.952782,0.000416
4,MLP,N_BASE_PATCH_ORGANIZATION,0.471839,0.080876,0.953111,0.000000
5,MLP,N_BASE_PLUS_XTASK_SMALL,0.470462,0.096129,0.946542,-0.001378


,model,representation,fold,AUPRC,AUROC,F1@0.5,MCC@0.5
0,MLP,N_BASE_PATCH_ORGANIZATION,1,0.574983,0.970843,0.354455,0.423127
1,MLP,N_BASE_PATCH_ORGANIZATION,2,0.536577,0.954191,0.375055,0.423874
2,MLP,N_BASE_PATCH_ORGANIZATION,3,0.400547,0.931114,0.310124,0.359102
3,MLP,N_BASE_PATCH_ORGANIZATION,4,0.451712,0.956107,0.425650,0.448528
4,MLP,N_BASE_PATCH_ORGANIZATION,5,0.395377,0.953300,0.370642,0.407591
5,MLP,N_BASE_PLUS_EVO,1,0.560913,0.965545,0.492986,0.517935
6,MLP,N_BASE_PLUS_EVO,2,0.546068,0.951047,0.359038,0.411942
7,MLP,N_BASE_PLUS_EVO,3,0.404669,0.934808,0.286773,0.348561
8,MLP,N_BASE_PLUS_EVO,4,0.474209,0.956506,0.414756,0.450072
9,MLP,N_BASE_PLUS_EVO,5,0.408961,0.954475,0.352376,0.404765



FINAL DECISION
{
  "base_dev_auprc": 0.4718393594050735,
  "best_representation": "N_BASE_PLUS_EVO",
  "best_dev_auprc": 0.4789641025500616,
  "gain_vs_base": 0.0071247431449881216,
  "promote_if_gain_ge_0.002": true,
  "EVO_available": true,
  "XTASK_small_available": true
}
Saved: /content/drive/MyDrive/Protein-binding-Phi/results/ADDIN_STANDALONE_NUCLEAR_EVO_XTASK


In [ ]:
from pathlib import Path
import gzip, time, json
import numpy as np
import pandas as pd

print('='*100)
print('NUCLEAR CLEAN V3 — EVO + TRUE SMALL -> NUCLEAR CROSS-TASK SCREEN')
print('='*100)

COUNTERPART = XTASK_ROOT / 'SMALL_DEV_OOF.csv.gz'
WAIT_SECONDS = 300
POLL_SECONDS = 5

t0 = time.time()

while (
    not COUNTERPART.exists()
    and
    (
        time.time()
        -
        t0
    )
    <
    WAIT_SECONDS
):

    print(
        f'Waiting for Small export... '
        f'{int(time.time()-t0)}s',
        end='\r'
    )

    time.sleep(
        POLL_SECONDS
    )


if not COUNTERPART.exists():
    raise FileNotFoundError(
        COUNTERPART
    )


def _read_maybe_gzip_csv(fp):

    with open(
        fp,
        'rb'
    ) as f:

        magic = f.read(2)

    if magic == b'\x1f\x8b':

        return pd.read_csv(
            fp
        )

    print(
        'WARNING: counterpart has .gz suffix but is plain CSV; '
        'reading safely as plain CSV.'
    )

    return pd.read_csv(
        fp,
        compression=None
    )


xdf = _read_maybe_gzip_csv(
    COUNTERPART
)


xdf[
    'prot_id'
] = xdf[
    'prot_id'
].astype(str)


xdf[
    'residue_index'
] = xdf[
    'residue_index'
].astype(int)


required = {
    'prot_id',
    'residue_index',
    'y_small',
    'prob_V1',
    'prob_MS',
}


missing = (
    required
    -
    set(
        xdf.columns
    )
)


if missing:

    raise RuntimeError(
        f'Wrong/old Small export. '
        f'Missing columns: {sorted(missing)}; '
        f'got {xdf.columns.tolist()}'
    )


if 'y_nuclear' in xdf.columns:

    raise RuntimeError(
        'WRONG FILE: Small export contains y_nuclear. '
        'Repair SMALL_DEV_OOF first.'
    )


if xdf.duplicated(
    [
        'prot_id',
        'residue_index'
    ]
).any():

    raise RuntimeError(
        'Duplicate (prot_id,residue_index) '
        'in Small OOF export.'
    )


expected_rows = int(
    train_df.sequence
    .str.len()
    .sum()
)


if len(xdf) != expected_rows:

    raise RuntimeError(
        f'Small OOF row count mismatch: '
        f'{len(xdf)} vs {expected_rows}'
    )


if set(
    xdf.prot_id
) != set(
    train_df
    .prot_id
    .astype(str)
):

    raise RuntimeError(
        'Small OOF protein IDs '
        'do not match DevSet1010.'
    )


print(
    'Small export integrity: PASS'
)

print(
    'rows:',
    len(xdf)
)

print(
    'columns:',
    xdf.columns.tolist()
)


# ============================================================
# Attach clean cross-task OOF:
# ONLY Small V1 + MS probabilities
# ============================================================

xmap = {

    pid:
        g.sort_values(
            'residue_index'
        )

    for pid, g
    in xdf.groupby(
        'prot_id',
        sort=False
    )
}


train_store = ctx.stores[
    STORE_KEY[
        'DevSet1010'
    ]
]


for r in train_df.itertuples():

    pid = str(
        r.prot_id
    )

    sub = xmap[
        pid
    ]

    L = len(
        r.sequence
    )


    if len(
        sub
    ) != L:

        raise RuntimeError(
            f'{pid}: Small OOF length '
            f'{len(sub)} != sequence length {L}'
        )


    if not np.array_equal(
        sub.residue_index.to_numpy(),
        np.arange(L)
    ):

        raise RuntimeError(
            f'{pid}: residue_index '
            'is not exactly 0..L-1'
        )


    arr = sub[
        [
            'prob_V1',
            'prob_MS'
        ]
    ].to_numpy(
        dtype=np.float32
    )


    if not np.all(
        np.isfinite(
            arr
        )
    ):

        raise RuntimeError(
            f'{pid}: non-finite '
            'Small OOF probability'
        )


    set_block(
        train_store,
        pid,
        'XTASK_SMALL2_CLEAN',
        arr
    )


print(
    'Attached XTASK_SMALL2_CLEAN: PASS'
)


# ============================================================
# EVO residual beyond ACCESS
# Dev-only because this is a representation screen.
# ============================================================

EVO_OK = bool(
    globals().get(
        'EVO_AVAILABLE',
        False
    )
)


if EVO_OK:

    for r in train_df.itertuples():

        E = np.asarray(
            get_block(
                train_store,
                r.prot_id,
                'EVO'
            ),
            dtype=np.float64
        )


        A = np.asarray(
            get_block(
                train_store,
                r.prot_id,
                'ACCESS'
            ),
            dtype=np.float64
        )


        if A.ndim == 1:

            A = A[
                :,
                None
            ]


        X = np.concatenate(
            [
                np.ones(
                    (
                        len(A),
                        1
                    )
                ),

                A,

                A * A,
            ],
            axis=1
        )


        if len(
            E
        ) > (
            X.shape[1]
            +
            2
        ):

            beta = np.linalg.lstsq(
                X,
                E,
                rcond=None
            )[0]

            R = (
                E
                -
                X @ beta
            )

        else:

            R = (
                E
                -
                E.mean(
                    axis=0,
                    keepdims=True
                )
            )


        sd = R.std(
            axis=0,
            keepdims=True
        )


        R = (
            R
            /
            np.where(
                sd < 1e-6,
                1.0,
                sd
            )
        )


        set_block(
            train_store,
            r.prot_id,
            'EVO_ACCESS_RES_CLEAN',
            R.astype(
                np.float32
            )
        )


    print(
        'Attached EVO_ACCESS_RES_CLEAN: PASS'
    )


# ============================================================
# CLEAN REPRESENTATION SCREEN
# ============================================================

BASE_BLOCKS = [
    'ESM',
    'ACCESS',
    'NA_PATCH3D40',
    'NA_TOPO12',
]


GRID = {

    'N_BASE_PATCH_ORGANIZATION_CLEANV3':
        BASE_BLOCKS,

    'N_BASE_PLUS_XTASK_SMALL2_CLEANV3':
        BASE_BLOCKS
        + [
            'XTASK_SMALL2_CLEAN'
        ],
}


if EVO_OK:

    GRID[
        'N_BASE_PLUS_EVO_CLEANV3'
    ] = (
        BASE_BLOCKS
        + [
            'EVO'
        ]
    )


    GRID[
        'N_BASE_PLUS_EVO_ACCESS_RES_CLEANV3'
    ] = (
        BASE_BLOCKS
        + [
            'EVO_ACCESS_RES_CLEAN'
        ]
    )


    GRID[
        'N_BASE_PLUS_EVO_XTASK_SMALL2_CLEANV3'
    ] = (
        BASE_BLOCKS
        + [
            'EVO',
            'XTASK_SMALL2_CLEAN'
        ]
    )


print(
    '\nFresh representations:'
)


for k, v in GRID.items():

    print(
        ' -',
        k,
        '=',
        v
    )


# New namespace -> no stale result reuse
RUN_VERSION = (
    'standalone_nuclear_evo_xtask_screen_'
    'CLEAN_V3_20260914'
)


cv, folds = suite.run_dev_cv_only(
    ctx,
    GRID,
    models=('MLP',),
    run_version=RUN_VERSION,
)


base_name = (
    'N_BASE_PATCH_ORGANIZATION_CLEANV3'
)


base = float(
    cv.loc[
        cv.representation.eq(
            base_name
        ),
        'CV_AUPRC'
    ].iloc[0]
)


cv = cv.copy()


cv[
    'delta_vs_base'
] = (
    cv[
        'CV_AUPRC'
    ]
    -
    base
)


cv = (
    cv
    .sort_values(
        'CV_AUPRC',
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print(
    '\nCLEAN V3 NUCLEAR RESULT'
)


display(
    cv
)

display(
    folds
)


cv.to_csv(
    ADDIN_ROOT
    / 'REPRESENTATION_SCREEN_CLEAN_V3.csv',
    index=False
)


folds.to_csv(
    ADDIN_ROOT
    / 'REPRESENTATION_SCREEN_FOLDS_CLEAN_V3.csv',
    index=False
)


best = cv.iloc[0]


xt = cv[
    cv.representation.eq(
        'N_BASE_PLUS_XTASK_SMALL2_CLEANV3'
    )
]


xt_score = float(
    xt.CV_AUPRC.iloc[0]
)


if EVO_OK:

    evo_row = cv[
        cv.representation.eq(
            'N_BASE_PLUS_EVO_CLEANV3'
        )
    ]

    evo_score = float(
        evo_row.CV_AUPRC.iloc[0]
    )


    combo_row = cv[
        cv.representation.eq(
            'N_BASE_PLUS_EVO_XTASK_SMALL2_CLEANV3'
        )
    ]

    combo_score = float(
        combo_row.CV_AUPRC.iloc[0]
    )

else:

    evo_score = None
    combo_score = None


decision = {

    'run_version':
        RUN_VERSION,

    'base_dev_auprc':
        base,

    'xtask_small2_dev_auprc':
        xt_score,

    'xtask_small2_gain_vs_base':
        xt_score
        -
        base,

    'evo_dev_auprc':
        evo_score,

    'evo_gain_vs_base':
        (
            None
            if evo_score is None
            else evo_score - base
        ),

    'evo_plus_xtask_dev_auprc':
        combo_score,

    'evo_plus_xtask_gain_vs_base':
        (
            None
            if combo_score is None
            else combo_score - base
        ),

    'best_representation':
        str(
            best.representation
        ),

    'best_dev_auprc':
        float(
            best.CV_AUPRC
        ),

    'best_gain_vs_base':
        float(
            best.delta_vs_base
        ),

    'promote_best_if_gain_ge_0.002':
        bool(
            best.delta_vs_base
            >=
            0.002
        ),

    'EVO_available':
        EVO_OK,

    'cross_task_source_columns': [
        'prob_V1',
        'prob_MS',
    ],
}


(
    ADDIN_ROOT
    / 'DECISION_CLEAN_V3.json'
).write_text(
    json.dumps(
        decision,
        indent=2
    )
)


print(
    '\nDECISION'
)

print(
    json.dumps(
        decision,
        indent=2
    )
)

print(
    'Saved:',
    ADDIN_ROOT
)

NUCLEAR CLEAN V3 — EVO + TRUE SMALL -> NUCLEAR CROSS-TASK SCREEN
Small export integrity: PASS
rows: 170181
columns: ['dataset', 'prot_id', 'residue_index', 'y_small', 'prob_FINAL', 'prob_V1', 'prob_MS']
Attached XTASK_SMALL2_CLEAN: PASS
Attached EVO_ACCESS_RES_CLEAN: PASS

Fresh representations:
 - N_BASE_PATCH_ORGANIZATION_CLEANV3 = ['ESM', 'ACCESS', 'NA_PATCH3D40', 'NA_TOPO12']
 - N_BASE_PLUS_XTASK_SMALL2_CLEANV3 = ['ESM', 'ACCESS', 'NA_PATCH3D40', 'NA_TOPO12', 'XTASK_SMALL2_CLEAN']
 - N_BASE_PLUS_EVO_CLEANV3 = ['ESM', 'ACCESS', 'NA_PATCH3D40', 'NA_TOPO12', 'EVO']
 - N_BASE_PLUS_EVO_ACCESS_RES_CLEANV3 = ['ESM', 'ACCESS', 'NA_PATCH3D40', 'NA_TOPO12', 'EVO_ACCESS_RES_CLEAN']
 - N_BASE_PLUS_EVO_XTASK_SMALL2_CLEANV3 = ['ESM', 'ACCESS', 'NA_PATCH3D40', 'NA_TOPO12', 'EVO', 'XTASK_SMALL2_CLEAN']
DONE MLP N_BASE_PATCH_ORGANIZATION_CLEANV3 1 AUPRC 0.5749834367678051
DONE MLP N_BASE_PATCH_ORGANIZATION_CLEANV3 2 AUPRC 0.5365769323700917
DONE MLP N_BASE_PATCH_ORGANIZATION_CLEANV3 3 AUPRC 0.40054

,model,representation,CV_AUPRC,CV_AUPRC_SD,CV_AUROC,delta_vs_base
0,MLP,N_BASE_PLUS_EVO_CLEANV3,0.478964,0.073589,0.952476,0.007125
1,MLP,N_BASE_PLUS_EVO_ACCESS_RES_CLEANV3,0.477783,0.075010,0.952847,0.005944
2,MLP,N_BASE_PLUS_EVO_XTASK_SMALL2_CLEANV3,0.476799,0.078186,0.953790,0.004960
3,MLP,N_BASE_PLUS_XTASK_SMALL2_CLEANV3,0.475690,0.084836,0.953790,0.003851
4,MLP,N_BASE_PATCH_ORGANIZATION_CLEANV3,0.471839,0.080876,0.953111,0.000000


,model,representation,fold,AUPRC,AUROC,F1@0.5,MCC@0.5
0,MLP,N_BASE_PATCH_ORGANIZATION_CLEANV3,1,0.574983,0.970843,0.354455,0.423127
1,MLP,N_BASE_PATCH_ORGANIZATION_CLEANV3,2,0.536577,0.954191,0.375055,0.423874
2,MLP,N_BASE_PATCH_ORGANIZATION_CLEANV3,3,0.400547,0.931114,0.310124,0.359102
3,MLP,N_BASE_PATCH_ORGANIZATION_CLEANV3,4,0.451712,0.956107,0.425650,0.448528
4,MLP,N_BASE_PATCH_ORGANIZATION_CLEANV3,5,0.395377,0.953300,0.370642,0.407591
5,MLP,N_BASE_PLUS_XTASK_SMALL2_CLEANV3,1,0.578890,0.966867,0.424779,0.473426
6,MLP,N_BASE_PLUS_XTASK_SMALL2_CLEANV3,2,0.551998,0.954283,0.402628,0.445775
7,MLP,N_BASE_PLUS_XTASK_SMALL2_CLEANV3,3,0.391070,0.935432,0.277858,0.339983
8,MLP,N_BASE_PLUS_XTASK_SMALL2_CLEANV3,4,0.446324,0.959718,0.336542,0.393852
9,MLP,N_BASE_PLUS_XTASK_SMALL2_CLEANV3,5,0.410171,0.952649,0.324506,0.385325



DECISION
{
  "run_version": "standalone_nuclear_evo_xtask_screen_CLEAN_V3_20260914",
  "base_dev_auprc": 0.4718393594050735,
  "xtask_small2_dev_auprc": 0.47569044707973,
  "xtask_small2_gain_vs_base": 0.003851087674656528,
  "evo_dev_auprc": 0.4789641025500616,
  "evo_gain_vs_base": 0.0071247431449881216,
  "evo_plus_xtask_dev_auprc": 0.4767991928908718,
  "evo_plus_xtask_gain_vs_base": 0.004959833485798315,
  "best_representation": "N_BASE_PLUS_EVO_CLEANV3",
  "best_dev_auprc": 0.4789641025500616,
  "best_gain_vs_base": 0.0071247431449881216,
  "promote_best_if_gain_ge_0.002": true,
  "EVO_available": true,
  "cross_task_source_columns": [
    "prob_V1",
    "prob_MS"
  ]
}
Saved: /content/drive/MyDrive/Protein-binding-Phi/results/ADDIN_STANDALONE_NUCLEAR_EVO_XTASK


In [ ]:
# ============================================================
# NUCLEAR FINAL ADD-IN — PATCHHG ATTENTION + EVO48
# matched Dev CV vs base PatchHG, then EVO+MS ensemble;
# external evaluation only if Dev promotion rule passes.
#
# Paste this ONE cell at the end of the current
# NUCLEAR_STANDALONE_EVO_XTASK_ADDIN runtime.
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    accuracy_score,
)

print("="*110)
print("NUCLEAR FINAL TEST — PATCHHG ATTENTION + EVO48")
print("="*110)

# ============================================================
# 0. REQUIREMENTS / CONFIG
# ============================================================

REQ = [
    "BASE", "ctx", "train_df", "SPLIT_DF", "STORE_KEY",
    "OFFICIAL_FOLDS", "get_block", "DEVICE",
    "V1_OOF", "MS_OOF", "Y_DEV",
]

missing = [
    x for x in REQ
    if x not in globals()
]

if missing:
    raise RuntimeError(
        "Run Nuclear standalone notebook through CLEAN V3 first. "
        "Missing: " + ", ".join(missing)
    )

if not bool(
    globals().get(
        "EVO_AVAILABLE",
        False
    )
):
    raise RuntimeError(
        "EVO_AVAILABLE=False. "
        "This final experiment requires EVO48."
    )

# confirm EVO exists on Dev
_ = get_block(
    ctx.stores[
        STORE_KEY["DevSet1010"]
    ],
    train_df.iloc[0].prot_id,
    "EVO"
)

FINAL_ROOT = (
    BASE
    / "results"
    / "ADDIN_STANDALONE_NUCLEAR_EVO_XTASK"
    / "FINAL_PATCHHG_EVO48"
)

FINAL_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

STATE_ROOT = (
    FINAL_ROOT
    / "states"
)

STATE_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

PATCH_CACHE = (
    BASE
    / "cache"
    / "nuclear_ultimate_v1"
    / "patch_hypergraph_v1"
)

SEED_FINAL = 2026

PATCH_LATENT = 192
PATCH_EPOCHS = 32
PATCH_PATIENCE = 6
PATCH_PROTEIN_BATCH = 4

PATCH_LR = 3e-4
PATCH_WEIGHT_DECAY = 1e-4

MIN_GAIN = 0.002

BASE_REP_NAME = (
    "PATCHHG_BASE_RECHECK_FINALV1"
)

EVO_REP_NAME = (
    "PATCHHG_EVO48_FINALV1"
)

BASE_MODALITY_SPEC = {
    "ESM": [
        "ESM"
    ],

    "ELECTRO": [
        "ACCESS"
    ],

    "PATCH": [
        "NA_PATCH3D40",
        "NA_TOPO12"
    ],
}

EVO_MODALITY_SPEC = {
    "ESM": [
        "ESM"
    ],

    "ELECTRO": [
        "ACCESS"
    ],

    "PATCH": [
        "NA_PATCH3D40",
        "NA_TOPO12"
    ],

    "BIO": [
        "EVO"
    ],
}


# ============================================================
# 1. HELPERS
# ============================================================

def concat_blocks_local(
    store,
    pid,
    blocks
):
    xs = [
        np.asarray(
            get_block(
                store,
                pid,
                b
            ),
            dtype=np.float32
        )
        for b in blocks
    ]

    lens = {
        x.shape[0]
        for x in xs
    }

    if len(lens) != 1:
        raise RuntimeError(
            f"{pid}: block length mismatch {blocks}"
        )

    return np.concatenate(
        xs,
        axis=1
    )


def load_patch_ram(
    split
):
    """
    Reuse exact V1 candidate-patch cache.
    No PDB parsing.
    No graph rebuilding.
    """

    df = SPLIT_DF[
        split
    ]

    outdir = (
        PATCH_CACHE
        / split
    )

    if not outdir.exists():
        raise FileNotFoundError(
            outdir
        )

    ram = {}

    for n, r in enumerate(
        df.itertuples(),
        start=1
    ):
        fp = (
            outdir
            / f"{r.prot_id}.npz"
        )

        if not fp.exists():
            raise FileNotFoundError(
                fp
            )

        z = np.load(
            fp
        )

        er = z[
            "edge_res"
        ].astype(
            np.int64
        )

        ep = z[
            "edge_patch"
        ].astype(
            np.int64
        )

        pf = z[
            "patch_feat"
        ].astype(
            np.float32
        )

        if (
            pf.ndim != 2
            or
            pf.shape[1] != 18
        ):
            raise RuntimeError(
                f"{split}/{r.prot_id}: "
                f"unexpected patch_feat {pf.shape}"
            )

        if len(er) != len(ep):
            raise RuntimeError(
                f"{split}/{r.prot_id}: "
                "edge length mismatch"
            )

        ram[
            (
                split,
                str(r.prot_id)
            )
        ] = (
            er,
            ep,
            pf
        )

        if n % 250 == 0:
            print(
                "patch cache",
                split,
                n,
                "/",
                len(df)
            )

    return ram


print(
    "\nLoading V1 patch graph cache for Dev only..."
)

PATCH_RAM_FINAL = load_patch_ram(
    "DevSet1010"
)

print(
    "Dev patch cache loaded:",
    len(PATCH_RAM_FINAL)
)


# ============================================================
# 2. DATASET / COLLATE
# ============================================================

class EvoPatchDataset(
    Dataset
):

    def __init__(
        self,
        split,
        df,
        modality_spec
    ):
        self.split = split

        self.df = df.reset_index(
            drop=True
        )

        self.store = ctx.stores[
            STORE_KEY[
                split
            ]
        ]

        self.modality_spec = (
            modality_spec
        )


    def __len__(
        self
    ):
        return len(
            self.df
        )


    def __getitem__(
        self,
        i
    ):
        r = self.df.iloc[
            i
        ]

        pid = str(
            r.prot_id
        )

        mods = {
            m:
                concat_blocks_local(
                    self.store,
                    pid,
                    blocks
                ).astype(
                    np.float32,
                    copy=False
                )

            for m, blocks
            in self.modality_spec.items()
        }

        (
            er,
            ep,
            pf
        ) = PATCH_RAM_FINAL[
            (
                self.split,
                pid
            )
        ]

        return {
            "pid":
                pid,

            "mods":
                mods,

            "y":
                np.asarray(
                    r.nuclear_binding,
                    dtype=np.float32
                ),

            "edge_res":
                er,

            "edge_patch":
                ep,

            "patch_feat":
                pf,
        }


def collate_patch_final(
    batch
):

    keys = sorted(
        batch[0][
            "mods"
        ]
    )

    mods = {
        k:
            torch.from_numpy(
                np.concatenate(
                    [
                        b[
                            "mods"
                        ][k]
                        for b
                        in batch
                    ],
                    axis=0
                )
            ).float()

        for k
        in keys
    }

    y = torch.from_numpy(
        np.concatenate(
            [
                b[
                    "y"
                ]
                for b
                in batch
            ]
        )
    ).float()

    patch_feat = []
    edge_res = []
    edge_patch = []

    roff = 0
    poff = 0

    for b in batch:

        L = len(
            b[
                "y"
            ]
        )

        P = len(
            b[
                "patch_feat"
            ]
        )

        if P:
            patch_feat.append(
                torch.from_numpy(
                    b[
                        "patch_feat"
                    ]
                ).float()
            )

        if len(
            b[
                "edge_res"
            ]
        ):

            edge_res.append(
                torch.from_numpy(
                    b[
                        "edge_res"
                    ]
                ).long()
                +
                roff
            )

            edge_patch.append(
                torch.from_numpy(
                    b[
                        "edge_patch"
                    ]
                ).long()
                +
                poff
            )

        roff += L
        poff += P


    patch_feat = (

        torch.cat(
            patch_feat,
            dim=0
        )

        if patch_feat

        else torch.zeros(
            (
                0,
                18
            ),
            dtype=torch.float32
        )
    )


    edge_res = (

        torch.cat(
            edge_res,
            dim=0
        )

        if edge_res

        else torch.zeros(
            (
                0,
            ),
            dtype=torch.long
        )
    )


    edge_patch = (

        torch.cat(
            edge_patch,
            dim=0
        )

        if edge_patch

        else torch.zeros(
            (
                0,
            ),
            dtype=torch.long
        )
    )


    return {
        "mods":
            mods,

        "y":
            y,

        "edge_res":
            edge_res,

        "edge_patch":
            edge_patch,

        "patch_feat":
            patch_feat,
    }


# ============================================================
# 3. SAME V1 ATTENTION PATCHHG
# ============================================================

class PatchResidueEncoderFinal(
    nn.Module
):

    def __init__(
        self,
        dims,
        latent=192,
        drop=0.2
    ):
        super().__init__()

        self.keys = sorted(
            dims
        )

        self.enc = nn.ModuleDict({

            k:
                nn.Sequential(
                    nn.LayerNorm(
                        d
                    ),
                    nn.Linear(
                        d,
                        latent
                    ),
                    nn.GELU(),
                    nn.Dropout(
                        drop
                    )
                )

            for k, d
            in dims.items()
        })


        self.attn = (
            nn.MultiheadAttention(
                latent,
                6,
                dropout=drop,
                batch_first=True
            )
        )


    def forward(
        self,
        x
    ):

        Z = torch.stack(
            [
                self.enc[
                    k
                ](
                    x[
                        k
                    ]
                )
                for k
                in self.keys
            ],
            dim=1
        )


        A, _ = self.attn(
            Z,
            Z,
            Z,
            need_weights=False
        )


        return (
            Z
            +
            A
        ).mean(
            dim=1
        )


class NuclearPatchHypergraphFinal(
    nn.Module
):

    def __init__(
        self,
        dims,
        latent=192,
        drop=0.2
    ):

        super().__init__()


        self.encoder = (
            PatchResidueEncoderFinal(
                dims,
                latent=latent,
                drop=drop
            )
        )


        self.patch_static = (
            nn.Sequential(
                nn.LayerNorm(
                    18
                ),
                nn.Linear(
                    18,
                    64
                ),
                nn.GELU()
            )
        )


        self.patch_mlp = (
            nn.Sequential(
                nn.Linear(
                    latent * 2
                    +
                    64,
                    latent
                ),
                nn.GELU(),
                nn.Dropout(
                    drop
                ),
                nn.Linear(
                    latent,
                    latent
                ),
                nn.GELU(),
            )
        )


        self.message_mlp = (
            nn.Sequential(
                nn.LayerNorm(
                    latent
                ),
                nn.Linear(
                    latent,
                    latent
                ),
                nn.GELU(),
                nn.Dropout(
                    drop
                ),
                nn.Linear(
                    latent,
                    latent
                )
            )
        )


        self.alpha_logit = (
            nn.Parameter(
                torch.tensor(
                    -2.944,
                    dtype=torch.float32
                )
            )
        )


        self.head = (
            nn.Sequential(
                nn.LayerNorm(
                    latent
                ),
                nn.Linear(
                    latent,
                    128
                ),
                nn.GELU(),
                nn.Dropout(
                    drop
                ),
                nn.Linear(
                    128,
                    1
                )
            )
        )


    @staticmethod
    def index_mean(
        x,
        index,
        n
    ):

        out = x.new_zeros(
            (
                n,
                x.shape[-1]
            )
        )

        cnt = x.new_zeros(
            (
                n,
                1
            )
        )

        if len(
            index
        ):

            out.index_add_(
                0,
                index,
                x
            )

            cnt.index_add_(
                0,
                index,
                x.new_ones(
                    (
                        len(index),
                        1
                    )
                )
            )


        return (
            out
            /
            cnt.clamp_min(
                1.0
            )
        )


    @staticmethod
    def index_max(
        x,
        index,
        n
    ):

        if n == 0:

            return x.new_zeros(
                (
                    0,
                    x.shape[-1]
                )
            )


        base = x.new_full(
            (
                n,
                x.shape[-1]
            ),
            -1e9
        )


        if len(
            index
        ):

            idx = index[
                :,
                None
            ].expand(
                -1,
                x.shape[-1]
            )


            out = (
                base.scatter_reduce(
                    0,
                    idx,
                    x,
                    reduce="amax",
                    include_self=True
                )
            )

        else:

            out = base


        return torch.where(
            out < -1e8,
            torch.zeros_like(
                out
            ),
            out
        )


    def forward(
        self,
        mods,
        edge_res,
        edge_patch,
        patch_feat
    ):

        h = self.encoder(
            mods
        )

        nres = h.shape[
            0
        ]

        npatch = patch_feat.shape[
            0
        ]


        alpha = torch.sigmoid(
            self.alpha_logit
        )


        if (
            npatch
            and
            edge_res.numel()
        ):

            mean = self.index_mean(
                h[
                    edge_res
                ],
                edge_patch,
                npatch
            )


            mx = self.index_max(
                h[
                    edge_res
                ],
                edge_patch,
                npatch
            )


            ps = self.patch_static(
                patch_feat
            )


            hp = self.patch_mlp(
                torch.cat(
                    [
                        mean,
                        mx,
                        ps
                    ],
                    dim=-1
                )
            )


            msg = self.index_mean(
                hp[
                    edge_patch
                ],
                edge_res,
                nres
            )


            incident = h.new_zeros(
                (
                    nres,
                    1
                )
            )


            incident.index_add_(
                0,
                edge_res,
                h.new_ones(
                    (
                        len(
                            edge_res
                        ),
                        1
                    )
                )
            )


            delta = (
                self.message_mlp(
                    msg
                )
            )


            delta = (
                delta
                *
                (
                    incident > 0
                ).float()
            )


            h = (
                h
                +
                alpha
                *
                delta
            )


        return self.head(
            h
        ).squeeze(
            -1
        )


def modality_dims(
    spec
):

    store = ctx.stores[
        STORE_KEY[
            "DevSet1010"
        ]
    ]

    r = train_df.iloc[
        0
    ]


    return {

        m:
            int(
                concat_blocks_local(
                    store,
                    r.prot_id,
                    blocks
                ).shape[
                    1
                ]
            )

        for m, blocks
        in spec.items()
    }


# ============================================================
# 4. TRAIN / PREDICT
# ============================================================

def move_batch_final(
    b
):

    return {

        "mods": {
            k:
                v.to(
                    DEVICE,
                    non_blocking=True
                )

            for k, v
            in b[
                "mods"
            ].items()
        },

        "y":
            b[
                "y"
            ].to(
                DEVICE,
                non_blocking=True
            ),

        "edge_res":
            b[
                "edge_res"
            ].to(
                DEVICE,
                non_blocking=True
            ),

        "edge_patch":
            b[
                "edge_patch"
            ].to(
                DEVICE,
                non_blocking=True
            ),

        "patch_feat":
            b[
                "patch_feat"
            ].to(
                DEVICE,
                non_blocking=True
            ),
    }


def predict_final(
    model,
    dataset
):

    loader = DataLoader(
        dataset,
        batch_size=PATCH_PROTEIN_BATCH,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        collate_fn=collate_patch_final
    )


    pp = []
    yy = []


    model.eval()


    with torch.no_grad():

        for b in loader:

            b = move_batch_final(
                b
            )


            p = torch.sigmoid(
                model(
                    b[
                        "mods"
                    ],
                    b[
                        "edge_res"
                    ],
                    b[
                        "edge_patch"
                    ],
                    b[
                        "patch_feat"
                    ]
                )
            )


            pp.append(
                p.detach()
                .cpu()
                .numpy()
            )


            yy.append(
                b[
                    "y"
                ].detach()
                .cpu()
                .numpy()
            )


    return (
        np.concatenate(
            pp
        ),
        np.concatenate(
            yy
        )
    )


def train_fold_final(
    rep_name,
    modality_spec,
    tr_ds,
    va_ds,
    fold
):

    torch.manual_seed(
        SEED_FINAL
        +
        100
        +
        fold
    )


    np.random.seed(
        SEED_FINAL
        +
        100
        +
        fold
    )


    dims = modality_dims(
        modality_spec
    )


    model = (
        NuclearPatchHypergraphFinal(
            dims,
            latent=PATCH_LATENT,
            drop=0.2
        )
        .to(
            DEVICE
        )
    )


    ytr = np.concatenate(
        tr_ds.df.nuclear_binding.apply(
            lambda x:
                np.asarray(
                    x,
                    dtype=np.float32
                )
        ).tolist()
    )


    npos = max(
        float(
            ytr.sum()
        ),
        1.0
    )


    nneg = max(
        float(
            len(
                ytr
            )
            -
            ytr.sum()
        ),
        1.0
    )


    criterion = (
        nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor(
                np.sqrt(
                    nneg
                    /
                    npos
                ),
                dtype=torch.float32,
                device=DEVICE
            )
        )
    )


    opt = torch.optim.AdamW(
        model.parameters(),
        lr=PATCH_LR,
        weight_decay=PATCH_WEIGHT_DECAY
    )


    loader = DataLoader(
        tr_ds,
        batch_size=PATCH_PROTEIN_BATCH,
        shuffle=True,
        num_workers=0,
        pin_memory=True,
        collate_fn=collate_patch_final
    )


    best_ap = -1.0
    best_state = None
    bad = 0


    for epoch in range(
        1,
        PATCH_EPOCHS
        +
        1
    ):

        model.train()

        losses = []


        for b in loader:

            b = move_batch_final(
                b
            )


            opt.zero_grad(
                set_to_none=True
            )


            logits = model(
                b[
                    "mods"
                ],
                b[
                    "edge_res"
                ],
                b[
                    "edge_patch"
                ],
                b[
                    "patch_feat"
                ]
            )


            loss = criterion(
                logits,
                b[
                    "y"
                ]
            )


            loss.backward()


            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                5.0
            )


            opt.step()


            losses.append(
                float(
                    loss.detach()
                    .cpu()
                )
            )


        vp, vy = predict_final(
            model,
            va_ds
        )


        val_ap = float(
            average_precision_score(
                vy,
                vp
            )
        )


        if (
            val_ap
            >
            best_ap
            +
            1e-5
        ):

            best_ap = val_ap


            best_state = {
                k:
                    v.detach()
                    .cpu()
                    .clone()

                for k, v
                in model.state_dict().items()
            }


            bad = 0

        else:

            bad += 1


        if (
            epoch == 1
            or
            epoch % 5 == 0
        ):

            print(
                rep_name,
                "fold",
                fold,
                "epoch",
                epoch,
                "loss",
                round(
                    float(
                        np.mean(
                            losses
                        )
                    ),
                    4
                ),
                "val_AP",
                round(
                    val_ap,
                    5
                ),
                "best",
                round(
                    best_ap,
                    5
                ),
                "alpha",
                round(
                    float(
                        torch.sigmoid(
                            model.alpha_logit
                        )
                        .detach()
                        .cpu()
                    ),
                    4
                )
            )


        if (
            bad
            >=
            PATCH_PATIENCE
        ):

            break


    if best_state is None:

        raise RuntimeError(
            f"{rep_name} fold {fold}: no state"
        )


    model.load_state_dict(
        best_state
    )


    return (
        model,
        best_state
    )


def run_cv_final(
    rep_name,
    modality_spec
):

    offsets = {}

    cursor = 0


    for r in train_df.itertuples():

        L = len(
            r.sequence
        )

        offsets[
            str(
                r.prot_id
            )
        ] = (
            cursor,
            cursor + L
        )

        cursor += L


    oof = np.full(
        cursor,
        np.nan,
        dtype=np.float32
    )


    rows = []

    states = {}


    rep_dir = (
        STATE_ROOT
        / rep_name
    )


    rep_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    for fold, val_ids in (
        OFFICIAL_FOLDS.items()
    ):

        val_set = set(
            map(
                str,
                val_ids
            )
        )


        tr_df = train_df[
            ~train_df.prot_id
            .astype(str)
            .isin(
                val_set
            )
        ].copy()


        va_df = train_df[
            train_df.prot_id
            .astype(str)
            .isin(
                val_set
            )
        ].copy()


        tr_ds = EvoPatchDataset(
            "DevSet1010",
            tr_df,
            modality_spec
        )


        va_ds = EvoPatchDataset(
            "DevSet1010",
            va_df,
            modality_spec
        )


        state_fp = (
            rep_dir
            / f"fold{fold}.pt"
        )


        pred_fp = (
            rep_dir
            / f"fold{fold}_val.npy"
        )


        meta_fp = (
            rep_dir
            / f"fold{fold}.json"
        )


        expected = int(
            va_df.sequence
            .str.len()
            .sum()
        )


        reuse = False


        if (
            state_fp.exists()
            and
            pred_fp.exists()
            and
            meta_fp.exists()
        ):

            try:

                meta = json.loads(
                    meta_fp.read_text()
                )


                pred = np.load(
                    pred_fp
                )


                reuse = (
                    meta.get(
                        "representation"
                    )
                    ==
                    rep_name

                    and

                    meta.get(
                        "n_val"
                    )
                    ==
                    expected

                    and

                    pred.shape
                    ==
                    (
                        expected,
                    )

                    and

                    np.all(
                        np.isfinite(
                            pred
                        )
                    )
                )

            except Exception:

                reuse = False


        if reuse:

            print(
                rep_name,
                "fold",
                fold,
                "REUSE"
            )


            state = torch.load(
                state_fp,
                map_location="cpu"
            )


            pred = np.load(
                pred_fp
            ).astype(
                np.float32
            )


        else:

            print(
                rep_name,
                "fold",
                fold,
                "TRAIN"
            )


            (
                model,
                state
            ) = train_fold_final(
                rep_name,
                modality_spec,
                tr_ds,
                va_ds,
                fold
            )


            pred, _ = predict_final(
                model,
                va_ds
            )


            torch.save(
                state,
                state_fp
            )


            np.save(
                pred_fp,
                pred.astype(
                    np.float32
                )
            )


            meta_fp.write_text(
                json.dumps(
                    {
                        "representation":
                            rep_name,

                        "modality_spec":
                            modality_spec,

                        "n_val":
                            expected,
                    },
                    indent=2
                )
            )


            del model


            if (
                DEVICE.type
                ==
                "cuda"
            ):

                torch.cuda.empty_cache()


        states[
            fold
        ] = state


        pos = 0


        for r in va_df.itertuples():

            L = len(
                r.sequence
            )


            a, b = offsets[
                str(
                    r.prot_id
                )
            ]


            oof[
                a:b
            ] = pred[
                pos:pos+L
            ]


            pos += L


        yv = np.concatenate(
            va_df.nuclear_binding.apply(
                lambda x:
                    np.asarray(
                        x,
                        dtype=np.float32
                    )
            ).tolist()
        )


        rows.append(
            {
                "representation":
                    rep_name,

                "fold":
                    fold,

                "AUPRC":
                    float(
                        average_precision_score(
                            yv,
                            pred
                        )
                    ),

                "n_val":
                    len(
                        pred
                    ),

                "n_positive":
                    int(
                        yv.sum()
                    ),
            }
        )


    if not np.all(
        np.isfinite(
            oof
        )
    ):

        raise RuntimeError(
            f"{rep_name}: incomplete OOF"
        )


    f = pd.DataFrame(
        rows
    )


    return (
        {
            "representation":
                rep_name,

            "CV_AUPRC_mean":
                float(
                    f.AUPRC.mean()
                ),

            "CV_AUPRC_std":
                float(
                    f.AUPRC.std(
                        ddof=1
                    )
                ),

            "OOF_AUPRC":
                float(
                    average_precision_score(
                        Y_DEV,
                        oof
                    )
                ),
        },

        f,

        oof,

        states
    )


# ============================================================
# 5. MATCHED BASE vs EVO PATCHHG
# ============================================================

print(
    "\n"
    +
    "="*110
)

print(
    "MATCHED BASE PATCHHG"
)

print(
    "="*110
)


(
    base_summary,
    base_folds,
    base_oof,
    base_states
) = run_cv_final(
    BASE_REP_NAME,
    BASE_MODALITY_SPEC
)


print(
    "\n"
    +
    "="*110
)

print(
    "PATCHHG + EVO48"
)

print(
    "="*110
)


(
    evo_summary,
    evo_folds,
    evo_oof,
    evo_states
) = run_cv_final(
    EVO_REP_NAME,
    EVO_MODALITY_SPEC
)


matched = pd.DataFrame(
    [
        base_summary,
        evo_summary
    ]
)


base_score = float(
    base_summary[
        "CV_AUPRC_mean"
    ]
)


evo_score = float(
    evo_summary[
        "CV_AUPRC_mean"
    ]
)


evo_gain = (
    evo_score
    -
    base_score
)


fold_cmp = (
    base_folds[
        [
            "fold",
            "AUPRC"
        ]
    ]
    .rename(
        columns={
            "AUPRC":
                "AUPRC_BASE"
        }
    )
    .merge(
        evo_folds[
            [
                "fold",
                "AUPRC"
            ]
        ]
        .rename(
            columns={
                "AUPRC":
                    "AUPRC_EVO"
            }
        ),
        on="fold"
    )
)


fold_cmp[
    "delta_EVO_minus_BASE"
] = (
    fold_cmp[
        "AUPRC_EVO"
    ]
    -
    fold_cmp[
        "AUPRC_BASE"
    ]
)


print(
    "\nMATCHED DEV RESULT"
)


display(
    matched
)

display(
    fold_cmp
)


print(
    "Base PatchHG mean AUPRC:",
    base_score
)


print(
    "EVO PatchHG mean AUPRC:",
    evo_score
)


print(
    "EVO gain:",
    evo_gain
)


print(
    "EVO wins folds:",
    int(
        (
            fold_cmp[
                "delta_EVO_minus_BASE"
            ]
            >
            0
        ).sum()
    ),
    "/5"
)


matched.to_csv(
    FINAL_ROOT
    / "MATCHED_PATCHHG_BASE_VS_EVO.csv",
    index=False
)


fold_cmp.to_csv(
    FINAL_ROOT
    / "MATCHED_PATCHHG_FOLDS.csv",
    index=False
)


# ============================================================
# 6. DEV-ONLY EVO-PATCHHG + EXISTING MS ENSEMBLE
# ============================================================

def fold_indices_final():

    offsets = {}

    cursor = 0


    for r in train_df.itertuples():

        L = len(
            r.sequence
        )


        offsets[
            str(
                r.prot_id
            )
        ] = (
            cursor,
            cursor + L
        )


        cursor += L


    out = {}


    for fold, ids in (
        OFFICIAL_FOLDS.items()
    ):

        idx = []


        for pid in ids:

            a, b = offsets[
                str(
                    pid
                )
            ]


            idx.extend(
                range(
                    a,
                    b
                )
            )


        out[
            fold
        ] = np.asarray(
            idx,
            dtype=np.int64
        )


    return out


FIDX_FINAL = (
    fold_indices_final()
)


ensemble_rows = []


for w in np.linspace(
    0.0,
    1.0,
    21
):

    p = (
        w
        *
        evo_oof

        +

        (
            1.0
            -
            w
        )
        *
        MS_OOF
    )


    aps = [
        float(
            average_precision_score(
                Y_DEV[
                    idx
                ],
                p[
                    idx
                ]
            )
        )

        for _, idx
        in sorted(
            FIDX_FINAL.items()
        )
    ]


    ensemble_rows.append(
        {
            "w_EVO_PATCHHG":
                float(
                    w
                ),

            "w_MS":
                float(
                    1.0
                    -
                    w
                ),

            "CV_AUPRC_mean":
                float(
                    np.mean(
                        aps
                    )
                ),

            "CV_AUPRC_std":
                float(
                    np.std(
                        aps,
                        ddof=1
                    )
                ),

            "OOF_AUPRC":
                float(
                    average_precision_score(
                        Y_DEV,
                        p
                    )
                ),

            **{
                f"fold{i+1}_AUPRC":
                    v

                for i, v
                in enumerate(
                    aps
                )
            },
        }
    )


ens_search = (
    pd.DataFrame(
        ensemble_rows
    )
    .sort_values(
        [
            "CV_AUPRC_mean",
            "w_EVO_PATCHHG"
        ],
        ascending=[
            False,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)


display(
    ens_search.head(
        10
    )
)


ens_search.to_csv(
    FINAL_ROOT
    / "EVO_PATCHHG_MS_WEIGHT_SEARCH.csv",
    index=False
)


best_ens = (
    ens_search.iloc[
        0
    ]
)


W_EVO = float(
    best_ens[
        "w_EVO_PATCHHG"
    ]
)


W_MS = float(
    best_ens[
        "w_MS"
    ]
)


NEW_FINAL_DEV = float(
    best_ens[
        "CV_AUPRC_mean"
    ]
)


NEW_FINAL_OOF = (
    W_EVO
    *
    evo_oof

    +

    W_MS
    *
    MS_OOF
)


# old final V1 + MS
old_rows = []


for w in np.linspace(
    0.0,
    1.0,
    21
):

    p = (
        w
        *
        V1_OOF

        +

        (
            1.0
            -
            w
        )
        *
        MS_OOF
    )


    aps = [
        float(
            average_precision_score(
                Y_DEV[
                    idx
                ],
                p[
                    idx
                ]
            )
        )

        for _, idx
        in sorted(
            FIDX_FINAL.items()
        )
    ]


    old_rows.append(
        float(
            np.mean(
                aps
            )
        )
    )


OLD_FINAL_DEV = float(
    np.max(
        old_rows
    )
)


gain_vs_old_final = (
    NEW_FINAL_DEV
    -
    OLD_FINAL_DEV
)


PROMOTE_EVO_MODEL = (
    evo_gain
    >=
    MIN_GAIN
)


PROMOTE_NEW_FINAL = (
    gain_vs_old_final
    >=
    MIN_GAIN
)


print(
    "\n"
    +
    "="*110
)

print(
    "DEV FINAL DECISION"
)

print(
    "="*110
)


print(
    "Matched EVO PatchHG gain vs matched base:",
    evo_gain
)


print(
    "Promote EVO architecture?",
    PROMOTE_EVO_MODEL
)


print(
    "Best new ensemble:",
    f"{W_EVO:.2f} EVO-PatchHG + {W_MS:.2f} MS"
)


print(
    "Old final V1+MS Dev:",
    OLD_FINAL_DEV
)


print(
    "New EVO-PatchHG+MS Dev:",
    NEW_FINAL_DEV
)


print(
    "Gain new vs old final:",
    gain_vs_old_final
)


print(
    "Promote new final ensemble?",
    PROMOTE_NEW_FINAL
)


# ============================================================
# 7. DEV-LOCKED MCC THRESHOLD
# ============================================================

def best_mcc_threshold_local(
    y,
    p
):

    best_mcc = -999.0
    best_t = 0.5


    for t in np.linspace(
        0.01,
        0.99,
        199
    ):

        pred = (
            p >= t
        ).astype(
            int
        )


        m = float(
            matthews_corrcoef(
                y,
                pred
            )
        )


        if m > best_mcc:

            best_mcc = m
            best_t = float(
                t
            )


    return (
        best_t,
        best_mcc
    )


NEW_THR, NEW_OOF_MCC = (
    best_mcc_threshold_local(
        Y_DEV,
        NEW_FINAL_OOF
    )
)


print(
    "New final frozen Dev-MCC threshold:",
    NEW_THR
)


print(
    "New final Dev OOF MCC:",
    NEW_OOF_MCC
)


# ============================================================
# 8. EXTERNAL ONLY IF DEV RULE PASSES
# ============================================================

def metrics_full(
    y,
    p,
    t
):

    pred = (
        p >= t
    ).astype(
        int
    )


    return {

        "AUPRC":
            float(
                average_precision_score(
                    y,
                    p
                )
            ),

        "AUROC":
            float(
                roc_auc_score(
                    y,
                    p
                )
            ),

        "Accuracy":
            float(
                accuracy_score(
                    y,
                    pred
                )
            ),

        "Precision_positive":
            float(
                precision_score(
                    y,
                    pred,
                    zero_division=0
                )
            ),

        "Recall_positive":
            float(
                recall_score(
                    y,
                    pred,
                    zero_division=0
                )
            ),

        "F1_positive":
            float(
                f1_score(
                    y,
                    pred,
                    zero_division=0
                )
            ),

        "MCC":
            float(
                matthews_corrcoef(
                    y,
                    pred
                )
            ),
    }


external_rows = []


if PROMOTE_NEW_FINAL:

    print(
        "\nDev rule passed. Running external evaluation..."
    )


    for split in [
        "TestSet300",
        "TestSetNew46"
    ]:

        df = SPLIT_DF[
            split
        ]


        store = ctx.stores[
            STORE_KEY[
                split
            ]
        ]


        try:

            _ = get_block(
                store,
                df.iloc[0].prot_id,
                "EVO"
            )

        except Exception as e:

            raise RuntimeError(
                f"{split}: EVO48 not attached. "
                "Attach cached EVO to external sets "
                "before rerunning this cell."
            ) from e


        ext_ram = load_patch_ram(
            split
        )


        PATCH_RAM_FINAL.update(
            ext_ram
        )


        ds = EvoPatchDataset(
            split,
            df,
            EVO_MODALITY_SPEC
        )


        evo_preds = []


        for fold, state in (
            evo_states.items()
        ):

            model = (
                NuclearPatchHypergraphFinal(
                    modality_dims(
                        EVO_MODALITY_SPEC
                    ),
                    latent=PATCH_LATENT,
                    drop=0.2
                )
                .to(
                    DEVICE
                )
            )


            model.load_state_dict(
                state
            )


            p, _ = predict_final(
                model,
                ds
            )


            evo_preds.append(
                p
            )


            del model


            if (
                DEVICE.type
                ==
                "cuda"
            ):

                torch.cuda.empty_cache()


        p_evo = np.mean(
            np.stack(
                evo_preds
            ),
            axis=0
        )


        paper_pred_fp = (
            BASE
            / "results"
            / "NUCLEAR_FINAL_PAPER_READY"
            / f"FINAL_RESIDUE_PREDICTIONS_{split}.csv.gz"
        )


        if not paper_pred_fp.exists():

            raise FileNotFoundError(
                paper_pred_fp
            )


        oldpred = pd.read_csv(
            paper_pred_fp
        )


        if "prob_MS" not in oldpred.columns:

            raise RuntimeError(
                f"{paper_pred_fp} lacks prob_MS"
            )


        p_ms = oldpred[
            "prob_MS"
        ].to_numpy(
            dtype=np.float64
        )


        if len(
            p_ms
        ) != len(
            p_evo
        ):

            raise RuntimeError(
                f"{split}: EVO/MS prediction length mismatch"
            )


        p_new = (
            W_EVO
            *
            p_evo

            +

            W_MS
            *
            p_ms
        )


        y = np.concatenate(
            df.nuclear_binding.apply(
                lambda x:
                    np.asarray(
                        x,
                        dtype=np.int8
                    )
            ).tolist()
        )


        for rule, thr in [
            (
                "fixed_0.5",
                0.5
            ),
            (
                "Dev_OOF_MCC",
                NEW_THR
            ),
        ]:

            m = metrics_full(
                y,
                p_new,
                thr
            )


            external_rows.append(
                {
                    "dataset":
                        split,

                    "method":
                        (
                            f"{W_EVO:.2f} EVO-PatchHG "
                            f"+ {W_MS:.2f} MS"
                        ),

                    "threshold_rule":
                        rule,

                    "threshold":
                        thr,

                    **m,
                }
            )


        out = oldpred[
            [
                "prot_id",
                "residue_index",
                "y_true"
            ]
        ].copy()


        out[
            "prob_EVO_PatchHG"
        ] = p_evo


        out[
            "prob_MS"
        ] = p_ms


        out[
            "prob_NEW_FINAL"
        ] = p_new


        out.to_csv(
            FINAL_ROOT
            / f"NEW_FINAL_PREDICTIONS_{split}.csv.gz",
            index=False
        )


    external_df = pd.DataFrame(
        external_rows
    )


    print(
        "\nEXTERNAL NEW FINAL"
    )


    display(
        external_df
    )


    external_df.to_csv(
        FINAL_ROOT
        / "EXTERNAL_NEW_FINAL.csv",
        index=False
    )


else:

    print(
        "\nNew ensemble did not beat old final by >=0.002 Dev AUPRC. "
        "External evaluation intentionally skipped."
    )


# ============================================================
# 9. FINAL DECISION
# ============================================================

decision = {

    "matched_base_patchhg_dev_auprc":
        base_score,

    "evo_patchhg_dev_auprc":
        evo_score,

    "evo_patchhg_gain_vs_matched_base":
        evo_gain,

    "evo_patchhg_fold_wins":
        int(
            (
                fold_cmp[
                    "delta_EVO_minus_BASE"
                ]
                >
                0
            ).sum()
        ),

    "promote_evo_architecture":
        bool(
            PROMOTE_EVO_MODEL
        ),

    "old_final_v1_ms_dev_auprc":
        OLD_FINAL_DEV,

    "new_w_EVO_PatchHG":
        W_EVO,

    "new_w_MS":
        W_MS,

    "new_final_dev_auprc":
        NEW_FINAL_DEV,

    "new_final_gain_vs_old_final":
        gain_vs_old_final,

    "promote_new_final":
        bool(
            PROMOTE_NEW_FINAL
        ),

    "new_final_dev_mcc_threshold":
        NEW_THR,

    "new_final_dev_oof_mcc":
        NEW_OOF_MCC,
}


(
    FINAL_ROOT
    / "FINAL_DECISION.json"
).write_text(
    json.dumps(
        decision,
        indent=2
    )
)


print(
    "\nFINAL DECISION"
)


print(
    json.dumps(
        decision,
        indent=2
    )
)


print(
    "\nSaved:",
    FINAL_ROOT
)

NUCLEAR FINAL TEST — PATCHHG ATTENTION + EVO48

Loading V1 patch graph cache for Dev only...
patch cache DevSet1010 250 / 1010
patch cache DevSet1010 500 / 1010
patch cache DevSet1010 750 / 1010
patch cache DevSet1010 1000 / 1010
Dev patch cache loaded: 1010

MATCHED BASE PATCHHG
PATCHHG_BASE_RECHECK_FINALV1 fold 1 TRAIN


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


PATCHHG_BASE_RECHECK_FINALV1 fold 1 epoch 1 loss 0.2636 val_AP 0.56773 best 0.56773 alpha 0.0499


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


PATCHHG_BASE_RECHECK_FINALV1 fold 1 epoch 5 loss 0.0763 val_AP 0.54551 best 0.59431 alpha 0.0501


...

,representation,CV_AUPRC_mean,CV_AUPRC_std,OOF_AUPRC
0,PATCHHG_BASE_RECHECK_FINALV1,0.501112,0.087375,0.467303
1,PATCHHG_EVO48_FINALV1,0.496018,0.089728,0.480063


,fold,AUPRC_BASE,AUPRC_EVO,delta_EVO_minus_BASE
0,1,0.604655,0.609684,0.005029
1,2,0.561322,0.550498,-0.010824
2,3,0.398081,0.409097,0.011017
3,4,0.513948,0.507663,-0.006285
4,5,0.427555,0.403145,-0.024410


Base PatchHG mean AUPRC: 0.5011121827505758
EVO PatchHG mean AUPRC: 0.49601752190643167
EVO gain: -0.005094660844144117
EVO wins folds: 2 /5


,w_EVO_PATCHHG,w_MS,CV_AUPRC_mean,CV_AUPRC_std,OOF_AUPRC,fold1_AUPRC,fold2_AUPRC,fold3_AUPRC,fold4_AUPRC,fold5_AUPRC
0,0.60,0.40,0.506615,0.091311,0.503589,0.622382,0.553596,0.414304,0.529447,0.413346
1,0.65,0.35,0.506442,0.091188,0.503271,0.622414,0.553797,0.414807,0.528053,0.413139
2,0.50,0.50,0.506399,0.091517,0.502784,0.621541,0.553437,0.412798,0.530925,0.413294
3,0.55,0.45,0.506355,0.091429,0.503335,0.621943,0.553235,0.413297,0.530001,0.413299
4,0.45,0.55,0.506089,0.091637,0.502000,0.621193,0.552942,0.411816,0.531269,0.413226
5,0.70,0.30,0.505982,0.091062,0.502611,0.622064,0.553775,0.414855,0.526436,0.412778
6,0.40,0.60,0.505524,0.091612,0.500807,0.620433,0.552561,0.411034,0.530748,0.412842
7,0.75,0.25,0.505438,0.090897,0.501615,0.621766,0.553740,0.414695,0.524261,0.412728
8,0.80,0.20,0.504881,0.090433,0.500003,0.620993,0.553580,0.414656,0.522072,0.413105
9,0.35,0.65,0.504766,0.091660,0.499166,0.619516,0.551759,0.409648,0.530460,0.412446



DEV FINAL DECISION
Matched EVO PatchHG gain vs matched base: -0.005094660844144117
Promote EVO architecture? False
Best new ensemble: 0.60 EVO-PatchHG + 0.40 MS
Old final V1+MS Dev: 0.5089611583669623
New EVO-PatchHG+MS Dev: 0.5066148983078826
Gain new vs old final: -0.0023462600590796745
Promote new final ensemble? False
New final frozen Dev-MCC threshold: 0.7177777777777777
New final Dev OOF MCC: 0.5234132665127836

New ensemble did not beat old final by >=0.002 Dev AUPRC. External evaluation intentionally skipped.

FINAL DECISION
{
  "matched_base_patchhg_dev_auprc": 0.5011121827505758,
  "evo_patchhg_dev_auprc": 0.49601752190643167,
  "evo_patchhg_gain_vs_matched_base": -0.005094660844144117,
  "evo_patchhg_fold_wins": 2,
  "promote_evo_architecture": false,
  "old_final_v1_ms_dev_auprc": 0.5089611583669623,
  "new_w_EVO_PatchHG": 0.6000000000000001,
  "new_w_MS": 0.3999999999999999,
  "new_final_dev_auprc": 0.5066148983078826,
  "new_final_gain_vs_old_final": -0.00234626005907967